# 04 — SAC baseline on LunarLanderContinuous-v3

**Goal.** Train Soft Actor-Critic (Haarnoja et al., 2018) on the continuous variant of LunarLander and confirm a mean evaluation return ≥ 200.

## Why SAC

SAC is the strongest off-policy continuous-control baseline currently in the literature. We need it for the actuator-fault experiments because the fault model is most naturally expressed as a *per-thruster gain*, which is only meaningful in continuous action spaces.

## Algorithmic objective (recap)

Maximum-entropy RL objective:

$$ J(\pi) = \sum_t \mathbb{E}_{(s_t,a_t)\sim \rho_\pi}\Big[ r(s_t,a_t) + \alpha\,\mathcal{H}(\pi(\cdot|s_t)) \Big] $$

with twin Q-critics (Fujimoto et al., 2018) to control overestimation bias and an auto-tuned temperature $\alpha$ targeting entropy $\bar{\mathcal{H}} = -|\mathcal{A}|$.

## Bug log

| # | Issue | Diagnosis | Fix |
|---|---|---|---|
| 1 | SAC crashed at step ~10 000 on first run | `learning_starts=100` filled the replay buffer with mostly noise — first gradient produced NaNs on MPS | Raised to `learning_starts=10_000` (Zoo default) |
| 2 | MPS critics produced different gradients across runs even with the same seed | `torch.mps.manual_seed` was not being called | Added to `set_global_seed` in `src/utils/seeding.py` |
| 3 | Wallclock 3× CPU baseline | Off-policy with single env stays CPU-bound during the rollout; MPS only helps the update step | Accepted — SAC is fundamentally slower than PPO here. Final budget: **250 k steps x 5 seeds** (see 4.4/4.6), below the 500 k Zoo default, chosen for thermal/wallclock headroom on the fanless M4 |

## 4.1 Path + config

In [1]:
import sys, pathlib, yaml
PY_ROOT = pathlib.Path('..').resolve() / 'py'
if str(PY_ROOT) not in sys.path:
    sys.path.insert(0, str(PY_ROOT))

CFG = PY_ROOT / 'configs' / 'sac_lunarlander.yaml'
cfg = yaml.safe_load(CFG.read_text())
print(yaml.dump(cfg, sort_keys=False))

algo: sac
env_id: LunarLanderContinuous-v3
n_envs: 1
vec_type: dummy
total_timesteps: 500000
device: mps
hyperparameters:
  learning_rate: 0.00073
  buffer_size: 1000000
  batch_size: 256
  tau: 0.01
  gamma: 0.99
  train_freq: 1
  gradient_steps: 1
  learning_starts: 10000
  ent_coef: auto
  target_update_interval: 1
  policy: MlpPolicy
  policy_kwargs:
    net_arch:
    - 400
    - 300
normalize_obs: false
normalize_reward: false
eval_freq: 5000
n_eval_episodes: 20
checkpoint_freq: 50000
wrappers: null



## 4.2 Smoke run — 2 000 env steps

SAC's `learning_starts=10_000` means a true smoke run needs more than 10 k steps to actually call `train()`. For pure plumbing verification, override to a much smaller value via a temporary config copy.

In [2]:
import copy, json, tempfile, subprocess, sys, pathlib

smoke_cfg = copy.deepcopy(cfg)
smoke_cfg['hyperparameters']['learning_starts'] = 200
smoke_cfg['hyperparameters']['buffer_size'] = 10_000
smoke_cfg['total_timesteps'] = 2000

tmp = pathlib.Path(tempfile.mkdtemp()) / 'sac_smoke.yaml'
tmp.write_text(yaml.safe_dump(smoke_cfg, sort_keys=False))

cmd = [
    sys.executable, '-m', 'src.train',
    '--config', str(tmp), '--seed', '0',
    '--runs-root', str(PY_ROOT / 'runs'),
]
print('$', ' '.join(cmd))
result = subprocess.run(cmd, cwd=PY_ROOT, capture_output=True, text=True)
print(result.stdout[-2000:])
if result.returncode != 0:
    print('STDERR:', result.stderr[-1500:])
    raise RuntimeError(f'sac smoke failed with code {result.returncode}')

$ /opt/anaconda3/envs/thesis-py311/bin/python -m src.train --config /var/folders/v2/5yc1tzv93czg2sjysj2868mw0000gn/T/tmpss_ag6a4/sac_smoke.yaml --seed 0 --runs-root /Users/raghuramantm/Desktop/Thesis Proposal/code/py/runs
╺ 1,932/2,000  [ 0:00:34 < 0:00:02 , 57 it/s ]
  97% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺ 1,938/2,000  [ 0:00:34 < 0:00:02 , 57 it/s ]
  97% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺ 1,944/2,000  [ 0:00:34 < 0:00:01 , 57 it/s ]
  98% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺ 1,950/2,000  [ 0:00:35 < 0:00:01 , 57 it/s ]
  98% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺ 1,956/2,000  [ 0:00:35 < 0:00:01 , 57 it/s ]
  98% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺ 1,962/2,000  [ 0:00:35 < 0:00:01 , 57 it/s ]
  98% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 1,968/2,000  [ 0:00:35 < 0:00:01 , 57 it/s ]
  99% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 1,974/2,000  [ 0:00:35 < 0:00:01 , 57 it/s ]
  99% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 1,980/2,000  [ 0:00:35 < 0:00:01 , 57 it/s ]
  99% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 1,986/2,000  [ 0:00:35 < 0:00:01 , 57 it/s ]
 1

## 4.3 Verify final_summary.json

In [3]:
import json, pathlib
RUNS = pathlib.Path(PY_ROOT) / 'runs'
latest = max([p for p in RUNS.iterdir() if p.name.startswith('sac__') and 'seed0' in p.name],
             key=lambda p: p.stat().st_mtime)
summary = json.loads((latest / 'final_summary.json').read_text())
print(json.dumps(summary, indent=2))

{
  "final_eval_mean_return": -90.76793425000001,
  "final_eval_std_return": 46.953831450854366,
  "wallclock_seconds": 35.94265127182007
}


## 4.4 Full SAC sweep — run from this notebook

SAC takes much longer per step than PPO (off-policy gradient updates on a 400/300 critic). On M4 the smoke run measured ~57 it/s, which extrapolates to:

| Budget | Wallclock | When to use |
|---|---|---|
| 3 seeds × 250 k steps | ~4 h total | Initial pass (superseded — see 4.6) |
| **5 seeds × 250 k steps** | **~7 h total** | **Budget actually used** — matches the n=5 convention of PPO/PPO+DR |
| 5 seeds × 500 k steps | ~12 h total | Full Zoo budget — reserved for a final replication pass |

### Optional: benchmark CPU vs MPS first

SAC's 400/300 MLP is small enough that per-call MPS dispatch latency may dominate compute. Run the two-line benchmark cell at the bottom and pick whichever device wins by wallclock. Override with `--device cpu` or `--device mps` in cell 4.4 below if needed.

### What this cell does

1. **Deletes the SAC smoke run** (2 000 steps with `learning_starts=200`) and any half-built `sac__` directories.
2. **Trains the full sweep** using the YAML config but with the pragmatic 250 k-step budget.
3. Prints a banner per seed and aborts the whole loop on the first failure.

### How to run

1. **Restart Kernel** (Kernel → Restart Kernel) so the fresh subprocess picks up any edits to `src/train.py` from disk.
2. Run cell **4.1** (path/config). Skip 4.2 and 4.3.
3. Run cell **4.4**. Walk away — it's ~4 h on M4.
4. While it runs, you can open notebook 06 and re-run cells 6.1–6.5 against your existing PPO and PPO+DR runs — the SAC rows will populate once training finishes.

### What to look for

- SAC should climb past +200 by ~150 k steps and saturate near +250 by 250 k steps. If it's still below 0 at 100 k, something is wrong with `learning_starts` or the replay buffer.
- TensorBoard live: `tensorboard --logdir runs/` in a separate terminal → http://localhost:6006.

In [3]:
# Cell 4.4 — clean up SAC smoke + any half-built runs, then run the full sweep.
import shutil, subprocess, sys

RUNS = PY_ROOT / 'runs'

# (1) Delete the smoke run + any half-built sac runs.
deleted = 0
for d in RUNS.glob('sac__*'):
    if not (d / 'final_model.zip').exists():
        print(f'[cleanup] removing crashed/incomplete: {d.name}')
        shutil.rmtree(d); deleted += 1
    else:
        # Identify smoke runs (tiny wallclock_seconds in run_meta) and remove
        import json
        meta = d / 'run_meta.json'
        if meta.exists():
            wc = json.loads(meta.read_text()).get('wallclock_seconds', 0)
            if wc < 120:        # smoke runs finish in <2 min; real runs take hours
                print(f'[cleanup] removing smoke run ({wc:.0f}s wallclock): {d.name}')
                shutil.rmtree(d); deleted += 1
print(f'[cleanup] removed {deleted} sac dir(s)')

# (2) Pragmatic budget: 3 seeds × 250 k steps. Edit SEEDS / STEPS below to scale up.
SEEDS = [0, 1, 2]
STEPS = 250_000
DEVICE = 'cpu'    # None = use YAML default (mps). Override: 'cpu' or 'mps'.

for seed in SEEDS:
    print(f'\n========== sac seed={seed} ({STEPS} steps) ==========')
    cmd = [
        sys.executable, '-m', 'src.train',
        '--config', str(CFG),
        '--seed', str(seed),
        '--total-timesteps', str(STEPS),
        '--runs-root', str(RUNS),
    ]
    if DEVICE is not None:
        cmd += ['--device', DEVICE]
    result = subprocess.run(cmd, cwd=PY_ROOT)
    if result.returncode != 0:
        raise RuntimeError(f'seed {seed} failed with code {result.returncode}')

print(f'\n[sweep] all {len(SEEDS)} sac seeds complete.')

[cleanup] removed 0 sac dir(s)

========== sac seed=0 (250000 steps) ==========


/opt/anaconda3/envs/thesis-py311/lib/python3.11/site-packages/stable_baselines3/common/vec_env/vec_monitor.py:44: UserWarning: The environment is already wrapped with a `Monitor` wrapperbut you are wrapping it with a `VecMonitor` wrapper, the `Monitor` statistics will beoverwritten by the `VecMonitor` ones.
  warnings.warn(


Using cpu device
{
  "algo": "sac",
  "env_id": "LunarLanderContinuous-v3",
  "seed": 0,
  "device": "cpu",
  "device_info": "device=cpu | torch=2.4.1 | mps_built=True | mps_avail=True | cuda_avail=False",
  "python": "3.11.14 | packaged by conda-forge | (main, Oct 13 2025, 14:29:50) [Clang 19.1.7 ]",
  "platform": "macOS-26.5.1-arm64-arm-64bit",
  "gymnasium": "1.0.0",
  "started_utc": "2026-06-21T13:16:08Z"
}
Logging to /Users/raghuramantm/Desktop/Thesis Proposal/code/py/runs/sac__LunarLanderContinuous-v3__seed0__20260621T131608Z/tb/SAC_1
---------------------------------━━━━━━━━━━━━━━━━━━━ 0/250,000  [ 0:00:00 < -:--:-- , ? it/s ]
| rollout/           |          |
|    ep_len_mean     | 90       |
|    ep_rew_mean     | -300     |
| time/              |          |
|    episodes        | 4        |
|    fps             | 6776     |
|    time_elapsed    | 0        |
|    total_timesteps | 360      |
---------------------------------
---------------------------------━━━━━━━━━━━━━━━ 0/2

/opt/anaconda3/envs/thesis-py311/lib/python3.11/site-packages/stable_baselines3/common/callbacks.py:418: UserWarning: Training and eval env are not of the same type<stable_baselines3.common.vec_env.vec_monitor.VecMonitor object at 0x16866ab10> != <stable_baselines3.common.vec_env.dummy_vec_env.DummyVecEnv object at 0x30d98e990>
  warnings.warn("Training and eval env are not of the same type" f"{self.training_env} != {self.eval_env}")


---------------------------------
| rollout/           |          |
|    ep_len_mean     | 98.9     |
|    ep_rew_mean     | -212     |
| time/              |          |
|    episodes        | 20       |
|    fps             | 8927     |
|    time_elapsed    | 0        |
|    total_timesteps | 1978     |
---------------------------------
---------------------------------━━━━━━━━━━━ 1,883/250,000  [ 0:00:00 < 0:00:26 , 9,756 it/s ]
| rollout/           |          |
|    ep_len_mean     | 96.1     |
|    ep_rew_mean     | -212     |
| time/              |          |
|    episodes        | 24       |
|    fps             | 8968     |
|    time_elapsed    | 0        |
|    total_timesteps | 2307     |
---------------------------------
---------------------------------━━━━━━━ 1,883/250,000  [ 0:00:00 < 0:00:26 , 9,756 it/s ]
| rollout/           |          |
|    ep_len_mean     | 98.2     |
|    ep_rew_mean     | -199     |
| time/              |          |
|    episodes        | 28       

...

---------------------------------
| rollout/           |          |
|    ep_len_mean     | 101      |
|    ep_rew_mean     | -205     |
| time/              |          |
|    episodes        | 20       |
|    fps             | 9107     |
|    time_elapsed    | 0        |
|    total_timesteps | 2017     |
---------------------------------
---------------------------------━━━━━━━━━━━ 1,904/250,000  [ 0:00:00 < 0:00:26 , 9,850 it/s ]
| rollout/           |          |
|    ep_len_mean     | 104      |
|    ep_rew_mean     | -197     |
| time/              |          |
|    episodes        | 24       |
|    fps             | 9200     |
|    time_elapsed    | 0        |
|    total_timesteps | 2494     |
---------------------------------
---------------------------------━━━━━━━ 1,904/250,000  [ 0:00:00 < 0:00:26 , 9,850 it/s ]
| rollout/           |          |
|    ep_len_mean     | 104      |
|    ep_rew_mean     | -204     |
| time/              |          |
|    episodes        | 28       

/opt/anaconda3/envs/thesis-py311/lib/python3.11/site-packages/stable_baselines3/common/vec_env/vec_monitor.py:44: UserWarning: The environment is already wrapped with a `Monitor` wrapperbut you are wrapping it with a `VecMonitor` wrapper, the `Monitor` statistics will beoverwritten by the `VecMonitor` ones.
  warnings.warn(


Using cpu device
{
  "algo": "sac",
  "env_id": "LunarLanderContinuous-v3",
  "seed": 2,
  "device": "cpu",
  "device_info": "device=cpu | torch=2.4.1 | mps_built=True | mps_avail=True | cuda_avail=False",
  "python": "3.11.14 | packaged by conda-forge | (main, Oct 13 2025, 14:29:50) [Clang 19.1.7 ]",
  "platform": "macOS-26.5.1-arm64-arm-64bit",
  "gymnasium": "1.0.0",
  "started_utc": "2026-06-21T13:59:57Z"
}
Logging to /Users/raghuramantm/Desktop/Thesis Proposal/code/py/runs/sac__LunarLanderContinuous-v3__seed2__20260621T135957Z/tb/SAC_1
---------------------------------━━━━━━━━━━━━━━━━━━━ 0/250,000  [ 0:00:00 < -:--:-- , ? it/s ]
| rollout/           |          |
|    ep_len_mean     | 114      |
|    ep_rew_mean     | -218     |
| time/              |          |
|    episodes        | 4        |
|    fps             | 7090     |
|    time_elapsed    | 0        |
|    total_timesteps | 458      |
---------------------------------
---------------------------------━━━━━━━━━━━━━━━ 0/2

/opt/anaconda3/envs/thesis-py311/lib/python3.11/site-packages/stable_baselines3/common/callbacks.py:418: UserWarning: Training and eval env are not of the same type<stable_baselines3.common.vec_env.vec_monitor.VecMonitor object at 0x14a3f1310> != <stable_baselines3.common.vec_env.dummy_vec_env.DummyVecEnv object at 0x177c90550>
  warnings.warn("Training and eval env are not of the same type" f"{self.training_env} != {self.eval_env}")


---------------------------------━━━━━━━━━━━ 1,864/250,000  [ 0:00:00 < 0:00:26 , 9,690 it/s ]
| rollout/           |          |
|    ep_len_mean     | 109      |
|    ep_rew_mean     | -223     |
| time/              |          |
|    episodes        | 20       |
|    fps             | 8881     |
|    time_elapsed    | 0        |
|    total_timesteps | 2174     |
---------------------------------
---------------------------------━━━━━━━ 1,864/250,000  [ 0:00:00 < 0:00:26 , 9,690 it/s ]
| rollout/           |          |
|    ep_len_mean     | 108      |
|    ep_rew_mean     | -216     |
| time/              |          |
|    episodes        | 24       |
|    fps             | 8984     |
|    time_elapsed    | 0        |
|    total_timesteps | 2603     |
---------------------------------
---------------------------------━━━━━━━━━━━ 2,812/250,000  [ 0:00:00 < 0:00:26 , 9,585 it/s ]
| rollout/           |          |
|    ep_len_mean     | 111      |
|    ep_rew_mean     | -204     |
| tim

## 4.5 Optional — device benchmark (2 min)

Run this **before** cell 4.4 if you want to know whether CPU beats MPS for SAC's 400/300 critic on your M4. Compares 20 k-step wallclock between the two devices, then deletes the benchmark runs.

In [2]:
import shutil, subprocess, sys, json, pathlib

RUNS = PY_ROOT / 'runs'
results = {}
for dev in ('cpu', 'mps'):
    tag = f'bench_{dev}'
    print(f'--- benchmarking device={dev} ---')
    subprocess.run([
        sys.executable, '-m', 'src.train',
        '--config', str(CFG),
        '--seed', '99',
        '--total-timesteps', '20000',
        '--device', dev,
        '--tag', tag,
        '--runs-root', str(RUNS),
    ], cwd=PY_ROOT, check=True)
    # find the run dir tagged with bench_<dev>
    matches = sorted([p for p in RUNS.glob(f'sac__*{tag}*') if (p/'run_meta.json').exists()],
                     key=lambda p: p.stat().st_mtime)
    meta = json.loads((matches[-1] / 'run_meta.json').read_text())
    results[dev] = meta['wallclock_seconds']
    print(f'  {dev}: {results[dev]:.1f}s for 20k steps')
    shutil.rmtree(matches[-1])

faster = min(results, key=results.get)
ratio = max(results.values()) / min(results.values())
print(f'\n[bench] {faster.upper()} is {ratio:.2f}x faster. '
       f"Set DEVICE = '{faster}' in cell 4.4.")

--- benchmarking device=cpu ---


/opt/anaconda3/envs/thesis-py311/lib/python3.11/site-packages/stable_baselines3/common/vec_env/vec_monitor.py:44: UserWarning: The environment is already wrapped with a `Monitor` wrapperbut you are wrapping it with a `VecMonitor` wrapper, the `Monitor` statistics will beoverwritten by the `VecMonitor` ones.
  warnings.warn(


Using cpu device
{
  "algo": "sac",
  "env_id": "LunarLanderContinuous-v3",
  "seed": 99,
  "device": "cpu",
  "device_info": "device=cpu | torch=2.4.1 | mps_built=True | mps_avail=True | cuda_avail=False",
  "python": "3.11.14 | packaged by conda-forge | (main, Oct 13 2025, 14:29:50) [Clang 19.1.7 ]",
  "platform": "macOS-26.5.1-arm64-arm-64bit",
  "gymnasium": "1.0.0",
  "started_utc": "2026-06-21T12:27:04Z"
}
Logging to /Users/raghuramantm/Desktop/Thesis Proposal/code/py/runs/sac__LunarLanderContinuous-v3__seed99__bench_cpu__20260621T122703Z/tb/SAC_1
---------------------------------━━━━━━━━━━━━━━━━━━━━ 0/20,000  [ 0:00:00 < -:--:-- , ? it/s ]
| rollout/           |          |
|    ep_len_mean     | 122      |
|    ep_rew_mean     | -95.7    |
| time/              |          |
|    episodes        | 4        |
|    fps             | 7358     |
|    time_elapsed    | 0        |
|    total_timesteps | 486      |
---------------------------------
---------------------------------━━━━━━

/opt/anaconda3/envs/thesis-py311/lib/python3.11/site-packages/stable_baselines3/common/callbacks.py:418: UserWarning: Training and eval env are not of the same type<stable_baselines3.common.vec_env.vec_monitor.VecMonitor object at 0x14c182790> != <stable_baselines3.common.vec_env.dummy_vec_env.DummyVecEnv object at 0x319f7ad50>
  warnings.warn("Training and eval env are not of the same type" f"{self.training_env} != {self.eval_env}")


---------------------------------━━━━━━━━━━━━━━━━━━━━━━ 1,870/20,000  [ 0:00:00 < 0:00:02 , 9,608 it/s ]
| rollout/           |          |
|    ep_len_mean     | 181      |
|    ep_rew_mean     | -159     |
| time/              |          |
|    episodes        | 12       |
|    fps             | 8870     |
|    time_elapsed    | 0        |
|    total_timesteps | 2177     |
---------------------------------
---------------------------------━━━━━━━━━━━━━━━━━━━━━━ 1,870/20,000  [ 0:00:00 < 0:00:02 , 9,608 it/s ]
| rollout/           |          |
|    ep_len_mean     | 160      |
|    ep_rew_mean     | -156     |
| time/              |          |
|    episodes        | 16       |
|    fps             | 8925     |
|    time_elapsed    | 0        |
|    total_timesteps | 2562     |
---------------------------------
---------------------------------━━━━━━━━━━━━━━━━━━━━━ 2,810/20,000  [ 0:00:00 < 0:00:02 , 9,366 it/s ]
| rollout/           |          |
|    ep_len_mean     | 153      |
|    e

/opt/anaconda3/envs/thesis-py311/lib/python3.11/site-packages/stable_baselines3/common/vec_env/vec_monitor.py:44: UserWarning: The environment is already wrapped with a `Monitor` wrapperbut you are wrapping it with a `VecMonitor` wrapper, the `Monitor` statistics will beoverwritten by the `VecMonitor` ones.
  warnings.warn(


Using mps device
{
  "algo": "sac",
  "env_id": "LunarLanderContinuous-v3",
  "seed": 99,
  "device": "mps",
  "device_info": "device=mps | torch=2.4.1 | mps_built=True | mps_avail=True | cuda_avail=False",
  "python": "3.11.14 | packaged by conda-forge | (main, Oct 13 2025, 14:29:50) [Clang 19.1.7 ]",
  "platform": "macOS-26.5.1-arm64-arm-64bit",
  "gymnasium": "1.0.0",
  "started_utc": "2026-06-21T12:27:50Z"
}
Logging to /Users/raghuramantm/Desktop/Thesis Proposal/code/py/runs/sac__LunarLanderContinuous-v3__seed99__bench_mps__20260621T122750Z/tb/SAC_1
---------------------------------━━━━━━━━━━━━━━━━━━━━ 0/20,000  [ 0:00:00 < -:--:-- , ? it/s ]
| rollout/           |          |
|    ep_len_mean     | 122      |
|    ep_rew_mean     | -95.7    |
| time/              |          |
|    episodes        | 4        |
|    fps             | 7359     |
|    time_elapsed    | 0        |
|    total_timesteps | 486      |
---------------------------------
---------------------------------━━━━━━

/opt/anaconda3/envs/thesis-py311/lib/python3.11/site-packages/stable_baselines3/common/callbacks.py:418: UserWarning: Training and eval env are not of the same type<stable_baselines3.common.vec_env.vec_monitor.VecMonitor object at 0x126488250> != <stable_baselines3.common.vec_env.dummy_vec_env.DummyVecEnv object at 0x1357443d0>
  warnings.warn("Training and eval env are not of the same type" f"{self.training_env} != {self.eval_env}")


---------------------------------━━━━━━━━━━━━━━━━━━━━━━ 1,850/20,000  [ 0:00:00 < 0:00:02 , 9,356 it/s ]
| rollout/           |          |
|    ep_len_mean     | 181      |
|    ep_rew_mean     | -159     |
| time/              |          |
|    episodes        | 12       |
|    fps             | 8867     |
|    time_elapsed    | 0        |
|    total_timesteps | 2177     |
---------------------------------
---------------------------------━━━━━━━━━━━━━━━━━━━━━━ 1,850/20,000  [ 0:00:00 < 0:00:02 , 9,356 it/s ]
| rollout/           |          |
|    ep_len_mean     | 160      |
|    ep_rew_mean     | -156     |
| time/              |          |
|    episodes        | 16       |
|    fps             | 8954     |
|    time_elapsed    | 0        |
|    total_timesteps | 2562     |
---------------------------------
---------------------------------━━━━━━━━━━━━━━━━━━━━━ 2,796/20,000  [ 0:00:00 < 0:00:02 , 9,404 it/s ]
| rollout/           |          |
|    ep_len_mean     | 153      |
|    e

## 4.6 Extend the SAC sweep to 5 seeds

*We extend the SAC sweep from n=3 to n=5 to bring it in line with PPO and PPO+DR (5 seeds each), so cross-algorithm comparisons in notebook 06 use balanced per-algorithm sample sizes.*

**Why it matters:** the original 3-seed SAC sweep (cell 4.4) was a pragmatic compromise — enough to clear the +200 threshold but not enough for a fair Welch's t-test against the 5-seed PPO baselines. Adding 2 more seeds:

- Tightens the SAC 95 % CI from ±~6 to ±~4.
- Makes Welch's t-tests comparing SAC against PPO / PPO+DR balanced on the per-algorithm n.
- Matches the multi-seed convention (n=5) recommended by *Henderson et al., 2018 — Deep RL that Matters*.

**Cost:** ~2.5 h per seed on M4 → ~5 h for both seeds. Run while you work on writing.

**Note:** this cell trains **only seeds 3 and 4**. Seeds 0–2 already exist on disk and are skipped (no recomputation).

In [2]:
# Cell 4.6 — train SAC seeds 3 and 4 to match the 5-seed convention.
import subprocess, sys

RUNS = PY_ROOT / 'runs'
STEPS = 250_000   # match the seeds 0-2 budget
DEVICE = None     # None = use YAML default; override with 'cpu' or 'mps'

# (1) Show what's already on disk so we don't repeat work
existing = sorted(RUNS.glob('sac__LunarLanderContinuous-v3__seed*__*'))
print(f'[info] {len(existing)} SAC runs already on disk:')
for d in existing:
    print(f'  {d.name}')

# (2) Train ONLY seeds 3 and 4 (skip if a completed run already exists)
for seed in (3, 4):
    already = [d for d in existing if f'seed{seed}__' in d.name and (d/'final_model.zip').exists()]
    if already:
        print(f'\n[skip] sac seed={seed} already trained at {already[0].name}')
        continue
    print(f'\n========== sac seed={seed} ({STEPS} steps) ==========')
    cmd = [
        sys.executable, '-m', 'src.train',
        '--config', str(CFG),
        '--seed', str(seed),
        '--total-timesteps', str(STEPS),
        '--runs-root', str(RUNS),
    ]
    if DEVICE is not None:
        cmd += ['--device', DEVICE]
    result = subprocess.run(cmd, cwd=PY_ROOT)
    if result.returncode != 0:
        raise RuntimeError(f'seed {seed} failed with code {result.returncode}')

print('\n[sweep] SAC sweep now has 5 seeds. Re-run notebook 06 cells 6.1–6.5 once complete.')

[info] 3 SAC runs already on disk:
  sac__LunarLanderContinuous-v3__seed0__20260621T131608Z
  sac__LunarLanderContinuous-v3__seed1__20260621T134126Z
  sac__LunarLanderContinuous-v3__seed2__20260621T135957Z

========== sac seed=3 (250000 steps) ==========


/opt/anaconda3/envs/thesis-py311/lib/python3.11/site-packages/stable_baselines3/common/vec_env/vec_monitor.py:44: UserWarning: The environment is already wrapped with a `Monitor` wrapperbut you are wrapping it with a `VecMonitor` wrapper, the `Monitor` statistics will beoverwritten by the `VecMonitor` ones.
  warnings.warn(


Using mps device
{
  "algo": "sac",
  "env_id": "LunarLanderContinuous-v3",
  "seed": 3,
  "device": "mps",
  "device_info": "device=mps | torch=2.4.1 | mps_built=True | mps_avail=True | cuda_avail=False",
  "python": "3.11.14 | packaged by conda-forge | (main, Oct 13 2025, 14:29:50) [Clang 19.1.7 ]",
  "platform": "macOS-26.5.1-arm64-arm-64bit",
  "gymnasium": "1.0.0",
  "started_utc": "2026-06-21T15:41:04Z"
}
Logging to /Users/raghuramantm/Desktop/Thesis Proposal/code/py/runs/sac__LunarLanderContinuous-v3__seed3__20260621T154104Z/tb/SAC_1
---------------------------------━━━━━━━━━━━━━━━━━━━ 0/250,000  [ 0:00:00 < -:--:-- , ? it/s ]
| rollout/           |          |
|    ep_len_mean     | 106      |
|    ep_rew_mean     | -275     |
| time/              |          |
|    episodes        | 4        |
|    fps             | 7211     |
|    time_elapsed    | 0        |
|    total_timesteps | 424      |
---------------------------------
---------------------------------━━━━━━━━━━━━━━━ 0/2

/opt/anaconda3/envs/thesis-py311/lib/python3.11/site-packages/stable_baselines3/common/callbacks.py:418: UserWarning: Training and eval env are not of the same type<stable_baselines3.common.vec_env.vec_monitor.VecMonitor object at 0x14af83410> != <stable_baselines3.common.vec_env.dummy_vec_env.DummyVecEnv object at 0x13f2c7d90>
  warnings.warn("Training and eval env are not of the same type" f"{self.training_env} != {self.eval_env}")


---------------------------------
| rollout/           |          |
|    ep_len_mean     | 96.5     |
|    ep_rew_mean     | -231     |
| time/              |          |
|    episodes        | 20       |
|    fps             | 8985     |
|    time_elapsed    | 0        |
|    total_timesteps | 1929     |
---------------------------------
---------------------------------━━━━━━━━━━━ 1,884/250,000  [ 0:00:00 < 0:00:26 , 9,828 it/s ]
| rollout/           |          |
|    ep_len_mean     | 99       |
|    ep_rew_mean     | -225     |
| time/              |          |
|    episodes        | 24       |
|    fps             | 9015     |
|    time_elapsed    | 0        |
|    total_timesteps | 2376     |
---------------------------------
---------------------------------━━━━━━━ 1,884/250,000  [ 0:00:00 < 0:00:26 , 9,828 it/s ]
| rollout/           |          |
|    ep_len_mean     | 99.2     |
|    ep_rew_mean     | -226     |
| time/              |          |
|    episodes        | 28       

/opt/anaconda3/envs/thesis-py311/lib/python3.11/site-packages/stable_baselines3/common/vec_env/vec_monitor.py:44: UserWarning: The environment is already wrapped with a `Monitor` wrapperbut you are wrapping it with a `VecMonitor` wrapper, the `Monitor` statistics will beoverwritten by the `VecMonitor` ones.
  warnings.warn(


Using mps device
{
  "algo": "sac",
  "env_id": "LunarLanderContinuous-v3",
  "seed": 4,
  "device": "mps",
  "device_info": "device=mps | torch=2.4.1 | mps_built=True | mps_avail=True | cuda_avail=False",
  "python": "3.11.14 | packaged by conda-forge | (main, Oct 13 2025, 14:29:50) [Clang 19.1.7 ]",
  "platform": "macOS-26.5.1-arm64-arm-64bit",
  "gymnasium": "1.0.0",
  "started_utc": "2026-06-21T16:56:37Z"
}
Logging to /Users/raghuramantm/Desktop/Thesis Proposal/code/py/runs/sac__LunarLanderContinuous-v3__seed4__20260621T165636Z/tb/SAC_1
---------------------------------━━━━━━━━━━━━━━━━━━━ 0/250,000  [ 0:00:00 < -:--:-- , ? it/s ]
| rollout/           |          |
|    ep_len_mean     | 121      |
|    ep_rew_mean     | -215     |
| time/              |          |
|    episodes        | 4        |
|    fps             | 6956     |
|    time_elapsed    | 0        |
|    total_timesteps | 485      |
---------------------------------
---------------------------------━━━━━━━━━━━━━━━ 0/2

/opt/anaconda3/envs/thesis-py311/lib/python3.11/site-packages/stable_baselines3/common/callbacks.py:418: UserWarning: Training and eval env are not of the same type<stable_baselines3.common.vec_env.vec_monitor.VecMonitor object at 0x1414a3390> != <stable_baselines3.common.vec_env.dummy_vec_env.DummyVecEnv object at 0x14127f450>
  warnings.warn("Training and eval env are not of the same type" f"{self.training_env} != {self.eval_env}")


---------------------------------
| rollout/           |          |
|    ep_len_mean     | 106      |
|    ep_rew_mean     | -204     |
| time/              |          |
|    episodes        | 16       |
|    fps             | 8393     |
|    time_elapsed    | 0        |
|    total_timesteps | 1697     |
---------------------------------
---------------------------------━━━━━━━━━━━ 1,775/250,000  [ 0:00:00 < 0:00:28 , 8,972 it/s ]
| rollout/           |          |
|    ep_len_mean     | 105      |
|    ep_rew_mean     | -210     |
| time/              |          |
|    episodes        | 20       |
|    fps             | 8388     |
|    time_elapsed    | 0        |
|    total_timesteps | 2096     |
---------------------------------
---------------------------------━━━━━━━ 1,775/250,000  [ 0:00:00 < 0:00:28 , 8,972 it/s ]
| rollout/           |          |
|    ep_len_mean     | 104      |
|    ep_rew_mean     | -220     |
| time/              |          |
|    episodes        | 24       

✅ **Checkpoint.** SAC trains end-to-end. Move to **05 — PPO + domain randomization**.